In [1]:
from pathlib import Path
import pandas as pd
import numpy as np


# 1. Find any CSV dataset from Kaggle or UCI

In [2]:
#2. Load it and print the shape
airbnb_data = pd.read_csv('data/raw/Airbnb_Open_Data.csv', low_memory=False)
df = airbnb_data

df_shape = df.shape

print(f'Dataframe rows: {df_shape[0]}')
print(f'Dataframe columns: {df_shape[1]}')

print("\n___FIRST 5 ROWS___")
display(df.head())

#print("\n___INFO___")
#df.info

Dataframe rows: 102599
Dataframe columns: 26

___FIRST 5 ROWS___


,id,NAME,host id,host_identity_verified,host name,neighbourhood group,neighbourhood,lat,long,country,...,service fee,minimum nights,number of reviews,last review,reviews per month,review rate number,calculated host listings count,availability 365,house_rules,license
0,1001254,Clean & quiet apt home by the park,80014485718,unconfirmed,Madaline,Brooklyn,Kensington,40.64749,-73.97237,United States,...,$193,10.0,9.0,10/19/2021,0.21,4.0,6.0,286.0,Clean up and treat the home the way you'd like...,NaN
1,1002102,Skylit Midtown Castle,52335172823,verified,Jenna,Manhattan,Midtown,40.75362,-73.98377,United States,...,$28,30.0,45.0,5/21/2022,0.38,4.0,2.0,228.0,Pet friendly but please confirm with me if the...,NaN
2,1002403,THE VILLAGE OF HARLEM....NEW YORK !,78829239556,NaN,Elise,Manhattan,Harlem,40.80902,-73.94190,United States,...,$124,3.0,0.0,NaN,NaN,5.0,1.0,352.0,"I encourage you to use my kitchen, cooking and...",NaN
3,1002755,NaN,85098326012,unconfirmed,Garry,Brooklyn,Clinton Hill,40.68514,-73.95976,United States,...,$74,30.0,270.0,7/5/2019,4.64,4.0,1.0,322.0,NaN,NaN
4,1003689,Entire Apt: Spacious Studio/Loft by central park,92037596077,verified,Lyndon,Manhattan,East Harlem,40.79851,-73.94399,United States,...,$41,10.0,9.0,11/19/2018,0.10,3.0,1.0,289.0,"Please no smoking in the house, porch or on th...",NaN


In [3]:
#3. Identify numerical, categorical, and missing columns

#fixing column names

df.columns = df.columns.str.lower().str.replace(' ','_')
print(df.columns.tolist())

def get_numerical_cols(data_frame):
    numerical_cols = data_frame.select_dtypes(include=["int64", "float64"]).columns.tolist()
    print(f'NUMERICAL COLUMNS count: {len(numerical_cols)}\n{numerical_cols}')
    return numerical_cols

def get_categorical_cols(data_frame):
    categorical_cols = data_frame.select_dtypes(include=["str" ,"object", "category"]).columns.tolist()
    print(f'\nCATEGORICAL COLUMNS count: {len(categorical_cols)}\n{categorical_cols}')
    return categorical_cols

def get_missing_cols(data_frame):
    missing_count = data_frame.isnull().sum()
    missing_cols = missing_count[missing_count > 0]
    print(f"\nMissing columns: \n-------------------------------------------\n{missing_cols}") if not missing_cols.empty else print("NO missing columns")
    return missing_cols
    

#numerical_col
num_cols = get_numerical_cols(df)

#categorical columns 
cat_cols = get_categorical_cols(df)

#Missing columns
miss_cols = get_missing_cols(df)



['id', 'name', 'host_id', 'host_identity_verified', 'host_name', 'neighbourhood_group', 'neighbourhood', 'lat', 'long', 'country', 'country_code', 'instant_bookable', 'cancellation_policy', 'room_type', 'construction_year', 'price', 'service_fee', 'minimum_nights', 'number_of_reviews', 'last_review', 'reviews_per_month', 'review_rate_number', 'calculated_host_listings_count', 'availability_365', 'house_rules', 'license']
NUMERICAL COLUMNS count: 11
['id', 'host_id', 'lat', 'long', 'construction_year', 'minimum_nights', 'number_of_reviews', 'reviews_per_month', 'review_rate_number', 'calculated_host_listings_count', 'availability_365']

CATEGORICAL COLUMNS count: 15
['name', 'host_identity_verified', 'host_name', 'neighbourhood_group', 'neighbourhood', 'country', 'country_code', 'instant_bookable', 'cancellation_policy', 'room_type', 'price', 'service_fee', 'last_review', 'house_rules', 'license']

Missing columns: 
-------------------------------------------
name                       

In [4]:
#4. Build a full data dictionary with descriptions

data_dict = pd.read_csv("data/raw/Airbnb_Open_Data_Dictionary.csv", delimiter=";",quotechar="'",low_memory=False)

filtered_dict = data_dict[data_dict["Field"].isin(df.columns)].copy()

empty_desc_rows = filtered_dict[filtered_dict["Description"].isnull()]

print(empty_desc_rows["Field"].tolist())



['neighbourhood']


In [5]:
manual_desc = {
    "neighbourhood": "The neighbourhood where the apartment is located.",
    "neighbourhood_group": "The larger neighbourhood group or borough.",
    "lat": "Latitude coordinate of the listing.",
    "long": "Longitude coordinate of the listing.",
    "country": "Country where the listing is located.",
    "country_code": "Country code of the listing.",
    "cancellation_policy": "The cancellation policy of the listing.",
    "construction_year": "The year when the property was constructed.",
    "service_fee": "The service fee charged for the listing.",
    "review_rate_number": "The rating score of the listing.",
    "house_rules": "Rules provided by the host for the guests."
}

filtered_dict["Description"] = filtered_dict["Description"].fillna(
    filtered_dict["Field"].map(manual_desc)
)

In [6]:
#Make a summary table for every column
description_dict = filtered_dict.set_index("Field")["Description"].to_dict()
description_dict.update(manual_desc)

column_summary = pd.DataFrame({
    "Column Name": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Missing Values": df.isnull().sum().values,
    "Unique Values": df.nunique().values,
    "Description": [description_dict.get(col, "No description") for col in df.columns]
})

display(column_summary)

,Column Name,Data Type,Missing Values,Unique Values,Description
0,id,int64,0,102058,Airbnb's unique identifier for the listing
1,name,str,250,61281,Name of the listing
2,host_id,int64,0,102057,Airbnb's unique identifier for the host/user
3,host_identity_verified,str,289,2,t=true; f=false
4,host_name,str,406,13190,Name of the host. Usually just the first name(s).
5,neighbourhood_group,str,29,7,The larger neighbourhood group or borough.
6,neighbourhood,str,16,224,The neighbourhood where the apartment is located.
7,lat,float64,8,21991,Latitude coordinate of the listing.
8,long,float64,8,17774,Longitude coordinate of the listing.
9,country,str,532,1,Country where the listing is located.


In [7]:
#5. Convert at least two columns to the correct type
money_columns = ["price", "service_fee"]

for col in money_columns:
    df[col] = df[col].str.replace("$", "", regex=False)
    df[col] = df[col].str.replace(",", "", regex=False)
    df[col] = pd.to_numeric(df[col], errors="coerce")

#The dates use month/day/year format
df["last_review"] = pd.to_datetime(df["last_review"], format="%m/%d/%Y", errors="coerce")

print("Data types after conversion:")
print(df[["price", "service_fee", "last_review"]].dtypes)

#Update the summary after conversion
column_summary["Data Type"] = df.dtypes.astype(str).values
column_summary["Missing Values"] = df.isnull().sum().values

display(column_summary[column_summary["Column Name"].isin(
    ["price", "service_fee", "last_review"]
)])

Data types after conversion:
price                 float64
service_fee           float64
last_review    datetime64[us]
dtype: object


,Column Name,Data Type,Missing Values,Unique Values,Description
15,price,float64,247,1151,daily price in local currency
16,service_fee,float64,273,231,The service fee charged for the listing.
19,last_review,datetime64[us],15893,2477,The date of the last/newest review


In [8]:
#6. Extract two meaningful insights

# insight 1: connection between price and service fee
price_fee_corr = df["price"].corr(df["service_fee"])
service_fee_percent = (df["service_fee"] / df["price"] * 100).median()

print("insight 1:  Price and service fee")
print(f"Correlation: {price_fee_corr:.3f}")
print(f"The median service fee is {service_fee_percent:.1f}% of the price.")
print("The almost perfect correlation shows that the service fee is based on the price.")

# insight 2: typical minimum stay for each room type
minimum_stay_by_room = (
    df.groupby("room_type")["minimum_nights"]
    .median()
    .sort_values(ascending=False)
)

print("\n insight 2: median minimum stay by room type")
print(minimum_stay_by_room)
print("Entire homes have the longest typical minimum stay, while hotel and shared rooms have the shortest.")

insight 1:  Price and service fee
Correlation: 1.000
The median service fee is 20.0% of the price.
The almost perfect correlation shows that the service fee is based on the price.

 insight 2: median minimum stay by room type
room_type
Entire home/apt    3.0
Private room       2.0
Hotel room         1.0
Shared room        1.0
Name: minimum_nights, dtype: float64
Entire homes have the longest typical minimum stay, while hotel and shared rooms have the shortest.


In [9]:
#Save the cleaned dataset
output_path = Path("data/processed/Airbnb_Open_Data_Cleaned.csv")

df.to_csv(output_path, index=False)

print(f" cleaned dataset saved to: {output_path}")

Cleaned dataset saved to: data/processed/Airbnb_Open_Data_Cleaned.csv
